In [1]:
!pip -q install "transformers>=4.40" datasets accelerate evaluate scikit-learn pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [5]:
%%bash
set -e
PROJECT=/content/liar_project
mkdir -p $PROJECT/src/{common,data,methods,eval} $PROJECT/configs $PROJECT/artifacts

cat > $PROJECT/configs/base.yaml <<'YAML'
seed: 42

data:
  name: ucsbnlp/liar
  text_col: statement
  label_col: label
  splits: [train, validation, test]
  limit_eval: 500        # 先小跑，后面你再改大

prompt:
  mode: instruction      # baseline | instruction | cot

generation:
  max_input_length: 256
  max_new_tokens: 4      # label-only建议=4；cot训练/评测时你再改大
  do_sample: false
  num_beams: 1

metrics:
  latency_samples: 200   # 用多少条样本测 latency
YAML

cat > $PROJECT/src/common/seed.py <<'PY'
import random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
PY

cat > $PROJECT/src/common/io.py <<'PY'
import json, os
from dataclasses import asdict
from typing import Any, Dict

import yaml

def read_yaml(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

def write_json(path: str, obj: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def file_size_mb(path: str) -> float:
    if not os.path.exists(path):
        return 0.0
    return os.path.getsize(path) / (1024 * 1024)
PY

cat > $PROJECT/src/common/prompts.py <<'PY'
def make_source(statement: str, mode: str) -> str:
    s = str(statement)
    mode = str(mode).strip().lower()
    if mode == "baseline":
        return s

    if mode == "instruction":
        return (
            "Instruction: Decide whether the following claim is true or false.\n"
            f'Statement: "{s}"\n'
            "Options: True or False.\n"
            "Output the label only."
        )

    if mode == "cot":
        return (
            "Instruction: Decide whether the following claim is true or false.\n"
            "Before the final decision, briefly explain your reasoning in 1-2 sentences.\n"
            f'Statement: "{s}"\n'
            "Options: True or False.\n"
            "Format:\n"
            "Reasoning: <short explanation>\n"
            "Final label: <True or False>."
        )

    raise ValueError("prompt.mode must be baseline | instruction | cot")
PY

cat > $PROJECT/src/common/parsing.py <<'PY'
import re
from typing import Optional, Tuple

_FINAL_LABEL_RE = re.compile(r"final\s*label\s*:\s*(true|false)\b", re.IGNORECASE)
_BARE_LABEL_RE  = re.compile(r"\b(true|false)\b", re.IGNORECASE)

def extract_binary_label(text: str) -> Tuple[Optional[int], str]:
    """
    Returns: (label_int, how)
      - label_int: 1 for True, 0 for False, None if unparsable
      - how: 'final_label' | 'bare_label' | 'none'
    """
    t = str(text).strip()
    m = _FINAL_LABEL_RE.search(t)
    if m:
        return (1 if m.group(1).lower() == "true" else 0), "final_label"

    m = _BARE_LABEL_RE.search(t)
    if m:
        return (1 if m.group(1).lower() == "true" else 0), "bare_label"

    return None, "none"
PY

cat > $PROJECT/src/common/generation.py <<'PY'
from typing import Any, Dict

def build_generate_kwargs(cfg: Dict[str, Any], for_eval: bool = True) -> Dict[str, Any]:
    g = cfg["generation"]
    # 论文公平性：评测默认强制 greedy（你以后想加 beam/sampling 可以专门做 ablation）
    if for_eval:
        return dict(
            do_sample=False,
            num_beams=1,
            max_new_tokens=int(g["max_new_tokens"]),
        )
    return dict(
        do_sample=bool(g.get("do_sample", False)),
        num_beams=int(g.get("num_beams", 1)),
        max_new_tokens=int(g["max_new_tokens"]),
    )
PY

cat > $PROJECT/src/common/metrics.py <<'PY'
import time
from typing import List, Optional, Tuple

import torch
from sklearn.metrics import f1_score, accuracy_score

def macro_f1(true_labels: List[int], pred_labels: List[int]) -> float:
    return float(f1_score(true_labels, pred_labels, average="macro"))

def accuracy(true_labels: List[int], pred_labels: List[int]) -> float:
    return float(accuracy_score(true_labels, pred_labels))

@torch.no_grad()
def measure_latency_ms(generate_one, prompts: List[str], n_samples: int = 200) -> float:
    """
    generate_one(prompt) -> str
    """
    total = min(n_samples, len(prompts))
    if total <= 0:
        return 0.0

    # warmup
    _ = generate_one(prompts[0])
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.time()
    for i in range(total):
        _ = generate_one(prompts[i])
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.time()
    return (t1 - t0) / total * 1000.0
PY

cat > $PROJECT/src/data/liar.py <<'PY'
from typing import Dict, Any
from datasets import load_dataset, DatasetDict

LABEL_TRUE  = {"true", "mostly-true", "half-true"}
LABEL_FALSE = {"false", "barely-true", "pants-fire"}

def _to_binary(label: str) -> int:
    s = str(label).strip().lower()
    if s in LABEL_TRUE:
        return 1
    if s in LABEL_FALSE:
        return 0
    raise ValueError(f"Unknown LIAR label: {label}")

def load_liar(cfg: Dict[str, Any]) -> DatasetDict:
    name = cfg["data"]["name"]
    ds = load_dataset(name)

    def mapper(ex):
        return {
            "statement": ex["statement"],
            "label_bin": _to_binary(ex["label"]),
        }

    # standard splits: train/validation/test
    out = DatasetDict()
    for sp in ["train", "validation", "test"]:
        out[sp] = ds[sp].map(mapper, remove_columns=ds[sp].column_names)
    return out
PY

cat > $PROJECT/src/methods/base.py <<'PY'
from abc import ABC, abstractmethod
from typing import Any, Dict, Tuple

class BaseMethod(ABC):
    name: str

    @abstractmethod
    def load_for_eval(self, model_name: str, cfg: Dict[str, Any]):
        ...

    def train(self, cfg: Dict[str, Any]) -> str:
        raise NotImplementedError("Training will be implemented next (SFT / CoT / Soft Prompt).")
PY

cat > $PROJECT/src/methods/sft.py <<'PY'
from typing import Any, Dict, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from .base import BaseMethod

class SFTMethod(BaseMethod):
    name = "sft"

    def load_for_eval(self, model_name: str, cfg: Dict[str, Any]):
        tok = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        model.eval()
        return model, tok, device
PY

cat > $PROJECT/src/methods/registry.py <<'PY'
from .sft import SFTMethod

def get_method(name: str):
    name = str(name).strip().lower()
    if name in ["sft", "instruction_sft", "cot_sft", "soft_prompt"]:
        # 先都走同一个“加载-评测壳”，训练我们下一步再接
        return SFTMethod()
    raise ValueError(f"Unknown method: {name}")
PY

cat > $PROJECT/src/eval/evaluator.py <<'PY'
from typing import Any, Dict, List
import os
import torch

from transformers import GenerationConfig

from src.common.prompts import make_source
from src.common.parsing import extract_binary_label
from src.common.generation import build_generate_kwargs
from src.common.metrics import macro_f1, accuracy, measure_latency_ms

class Evaluator:
    def __init__(self, cfg: Dict[str, Any]):
        self.cfg = cfg

    @torch.no_grad()
    def evaluate(self, model, tok, device: str, dataset, split: str = "validation") -> Dict[str, Any]:
        limit = int(self.cfg["data"].get("limit_eval", 0))
        rows = dataset[split]
        if limit > 0:
            rows = rows.select(range(min(limit, len(rows))))

        prompt_mode = self.cfg["prompt"]["mode"]
        gen_kwargs = build_generate_kwargs(self.cfg, for_eval=True)

        true_labels: List[int] = []
        pred_labels: List[int] = []
        unparsable = 0
        used_prompts: List[str] = []

        def generate_one(prompt: str) -> str:
            inputs = tok(prompt, return_tensors="pt", truncation=True,
                         max_length=int(self.cfg["generation"]["max_input_length"])).to(device)
            out = model.generate(**inputs, **gen_kwargs)
            return tok.decode(out[0], skip_special_tokens=True)

        for ex in rows:
            prompt = make_source(ex["statement"], prompt_mode)
            used_prompts.append(prompt)

            text = generate_one(prompt)
            pred, how = extract_binary_label(text)
            true = int(ex["label_bin"])

            true_labels.append(true)
            if pred is None:
                unparsable += 1
                pred_labels.append(0)  # 占位，不影响 unparsable_rate 报告；你也可以改成跳过
            else:
                pred_labels.append(int(pred))

        f1 = macro_f1(true_labels, pred_labels)
        acc = accuracy(true_labels, pred_labels)

        lat_ms = measure_latency_ms(
            generate_one,
            used_prompts,
            n_samples=int(self.cfg["metrics"].get("latency_samples", 200))
        )

        return dict(
            split=split,
            n=len(true_labels),
            prompt_mode=prompt_mode,
            macro_f1=f1,
            accuracy=acc,
            mean_latency_ms=lat_ms,
            unparsable_rate=unparsable / max(1, len(true_labels)),
            generation=gen_kwargs
        )
PY

cat > $PROJECT/run.py <<'PY'
import argparse, os, time
from datetime import datetime

from src.common.io import read_yaml, ensure_dir, write_json
from src.common.seed import set_seed
from src.data.liar import load_liar
from src.methods.registry import get_method
from src.eval.evaluator import Evaluator

def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)

    ap_eval = sub.add_parser("eval")
    ap_eval.add_argument("--config", default="configs/base.yaml")
    ap_eval.add_argument("--method", default="sft")
    ap_eval.add_argument("--model", default="google/flan-t5-small")
    ap_eval.add_argument("--split", default="validation", choices=["train","validation","test"])
    ap_eval.add_argument("--run_name", default="")

    args = ap.parse_args()

    cfg_path = args.config if os.path.isabs(args.config) else os.path.join(os.path.dirname(__file__), args.config)
    cfg = read_yaml(cfg_path)

    set_seed(int(cfg.get("seed", 42)))

    ds = load_liar(cfg)
    method = get_method(args.method)
    model, tok, device = method.load_for_eval(args.model, cfg)

    evaluator = Evaluator(cfg)
    result = evaluator.evaluate(model, tok, device, ds, split=args.split)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = args.run_name.strip() or f"{args.method}_{args.model.replace('/','-')}_{cfg['prompt']['mode']}_{args.split}_{ts}"
    out_dir = ensure_dir(os.path.join(os.path.dirname(__file__), "artifacts", run_name))

    write_json(os.path.join(out_dir, "result.json"), result)
    write_json(os.path.join(out_dir, "manifest.json"), dict(
        run_name=run_name,
        method=args.method,
        model=args.model,
        config_path=args.config,
        split=args.split
    ))

    print("Saved:", out_dir)
    print(result)

if __name__ == "__main__":
    main()
PY

echo "Scaffold created at: $PROJECT"
ls -R $PROJECT | sed -n '1,200p'


Scaffold created at: /content/liar_project
/content/liar_project:
artifacts
configs
run.py
src

/content/liar_project/artifacts:

/content/liar_project/configs:
base.yaml

/content/liar_project/src:
common
data
eval
methods

/content/liar_project/src/common:
generation.py
io.py
metrics.py
parsing.py
prompts.py
__pycache__
seed.py

/content/liar_project/src/common/__pycache__:
generation.cpython-312.pyc
io.cpython-312.pyc
metrics.cpython-312.pyc
parsing.cpython-312.pyc
prompts.cpython-312.pyc
seed.cpython-312.pyc

/content/liar_project/src/data:
liar.py
__pycache__

/content/liar_project/src/data/__pycache__:
liar.cpython-312.pyc

/content/liar_project/src/eval:
evaluator.py
__pycache__

/content/liar_project/src/eval/__pycache__:
evaluator.cpython-312.pyc

/content/liar_project/src/methods:
base.py
__pycache__
registry.py
sft.py

/content/liar_project/src/methods/__pycache__:
base.cpython-312.pyc
registry.cpython-312.pyc
sft.cpython-312.pyc


In [6]:
%cd /content/liar_project
!python run.py eval --model google/flan-t5-small --method sft --split validation


/content/liar_project
Traceback (most recent call last):
  File "/content/liar_project/run.py", line 7, in <module>
    from src.methods.registry import get_method
  File "/content/liar_project/src/methods/registry.py", line 1, in <module>
    from .sft import SFTMethod
  File "/content/liar_project/src/methods/sft.py", line 3, in <module>
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
  File "<frozen importlib._bootstrap>", line 1412, in _handle_fromlist
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2317, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2345, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__ini

In [7]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/liar_project_backup
!cp -r /content/liar_project/artifacts /content/drive/MyDrive/liar_project_backup/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!pip -q install "transformers>=4.40" datasets accelerate evaluate scikit-learn pyyaml peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [2]:
%%bash
set -e
PROJECT=/content/liar_project_v2
mkdir -p $PROJECT/src/{common,data,methods,eval} $PROJECT/configs $PROJECT/artifacts $PROJECT/cache

cat > $PROJECT/configs/base.yaml <<'YAML'
seed: 42

data:
  name: ucsbnlp/liar
  splits: [train, validation, test]
  limit_eval: 500

label_map:
  true:  ["true", "mostly-true", "half-true"]
  false: ["false", "barely-true", "pants-fire"]

prompt:
  # 默认：不同方法用不同 prompt（sft/soft_prompt=instruction；cot=cot）
  # 你想强行统一也可以在命令行覆盖
  mode: instruction   # instruction | cot | baseline

generation:
  max_input_length: 256
  max_new_tokens: 64       # 为兼容 CoT，统一设大（符合你README“统一decode”）:contentReference[oaicite:1]{index=1}
  do_sample: false
  num_beams: 1

train:
  lr: 2e-4
  epochs: 3
  per_device_train_batch_size: 8
  per_device_eval_batch_size: 16
  grad_accum: 4
  weight_decay: 0.0
  fp16: true
  logging_steps: 50
  eval_steps: 400
  save_steps: 400
  max_train_samples: 0       # 0表示不截断
  max_eval_samples: 0

soft_prompt:
  m: 30   # num_virtual_tokens (you will sweep 20~50)

cot:
  # CoT SFT 需要 reasoning。我们用“教师模型”生成 reasoning（缓存到cache）再训练。
  teacher_flan: google/flan-t5-base
  teacher_bart: facebook/bart-large
YAML

cat > $PROJECT/src/common/seed.py <<'PY'
import random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
PY

cat > $PROJECT/src/common/io.py <<'PY'
import json, os
from typing import Any, Dict
import yaml

def read_yaml(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

def write_json(path: str, obj: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def file_size_mb(path: str) -> float:
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0.0

def dir_size_mb(root: str) -> float:
    total = 0
    for dp, _, fns in os.walk(root):
        for fn in fns:
            p = os.path.join(dp, fn)
            total += os.path.getsize(p)
    return total / (1024 * 1024)
PY

cat > $PROJECT/src/common/prompts.py <<'PY'
def make_source(statement: str, mode: str) -> str:
    s = str(statement)
    mode = str(mode).strip().lower()

    if mode == "baseline":
        return s

    if mode == "instruction":
        return (
            "Instruction: Decide whether the following claim is true or false.\n"
            f'Statement: "{s}"\n'
            "Options: True or False.\n"
            "Output the label only."
        )

    if mode == "cot":
        return (
            "Instruction: Decide whether the following claim is true or false.\n"
            "Before the final decision, briefly explain your reasoning in 1-2 sentences.\n"
            f'Statement: "{s}"\n'
            "Options: True or False.\n"
            "Format:\n"
            "Reasoning: <short explanation>\n"
            "Final label: <True or False>."
        )

    raise ValueError("prompt.mode must be baseline|instruction|cot")

def make_target_label_only(label_bin: int) -> str:
    return "True" if int(label_bin) == 1 else "False"

def make_target_cot(reasoning: str, label_bin: int) -> str:
    # 训练目标固定用 Final label:（与你 preference 一致）
    lab = "True" if int(label_bin) == 1 else "False"
    r = str(reasoning).strip()
    if not r:
        r = "Reasoning: (omitted)"
    # 强制格式稳定，方便解析和公平评测
    if not r.lower().startswith("reasoning:"):
        r = "Reasoning: " + r
    return f"{r}\nFinal label: {lab}"
PY

cat > $PROJECT/src/common/parsing.py <<'PY'
import re
from typing import Optional, Tuple

_FINAL_LABEL_RE = re.compile(r"final\s*label\s*:\s*(true|false)\b", re.IGNORECASE)
_BARE_LABEL_RE  = re.compile(r"\b(true|false)\b", re.IGNORECASE)

def extract_binary_label(text: str) -> Tuple[Optional[int], str]:
    t = str(text).strip()

    m = _FINAL_LABEL_RE.search(t)
    if m:
        return (1 if m.group(1).lower() == "true" else 0), "final_label"

    m = _BARE_LABEL_RE.search(t)
    if m:
        return (1 if m.group(1).lower() == "true" else 0), "bare_label"

    return None, "none"
PY

cat > $PROJECT/src/common/generation.py <<'PY'
from typing import Any, Dict

def build_generate_kwargs(cfg: Dict[str, Any], for_eval: bool = True) -> Dict[str, Any]:
    g = cfg["generation"]
    # 主结果强制 greedy（你选了A）
    if for_eval:
        return dict(
            do_sample=False,
            num_beams=1,
            max_new_tokens=int(g["max_new_tokens"])
        )
    return dict(
        do_sample=bool(g.get("do_sample", False)),
        num_beams=int(g.get("num_beams", 1)),
        max_new_tokens=int(g["max_new_tokens"])
    )
PY

cat > $PROJECT/src/common/metrics.py <<'PY'
import time
from typing import List, Callable
import torch
from sklearn.metrics import f1_score, accuracy_score

def macro_f1(true_labels: List[int], pred_labels: List[int]) -> float:
    return float(f1_score(true_labels, pred_labels, average="macro"))

def accuracy(true_labels: List[int], pred_labels: List[int]) -> float:
    return float(accuracy_score(true_labels, pred_labels))

@torch.no_grad()
def measure_latency_ms(generate_one: Callable[[str], str], prompts: List[str], n_samples: int = 200) -> float:
    total = min(n_samples, len(prompts))
    if total <= 0:
        return 0.0

    _ = generate_one(prompts[0])  # warmup
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.time()
    for i in range(total):
        _ = generate_one(prompts[i])
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.time()
    return (t1 - t0) / total * 1000.0
PY

cat > $PROJECT/src/data/liar.py <<'PY'
from typing import Any, Dict
from datasets import load_dataset, DatasetDict

def load_liar(cfg: Dict[str, Any]) -> DatasetDict:
    ds = load_dataset(cfg["data"]["name"])

    true_set = set([s.lower() for s in cfg["label_map"]["true"]])
    false_set = set([s.lower() for s in cfg["label_map"]["false"]])

    def to_bin(label: str) -> int:
        s = str(label).strip().lower()
        if s in true_set:
            return 1
        if s in false_set:
            return 0
        raise ValueError(f"Unknown LIAR label: {label}")

    def mapper(ex):
        return {
            "statement": ex["statement"],
            "label_bin": to_bin(ex["label"]),
        }

    out = DatasetDict()
    for sp in ["train", "validation", "test"]:
        out[sp] = ds[sp].map(mapper, remove_columns=ds[sp].column_names)
    return out
PY

cat > $PROJECT/src/methods/base.py <<'PY'
from abc import ABC, abstractmethod
from typing import Any, Dict, Tuple

class BaseMethod(ABC):
    name: str

    @abstractmethod
    def train(self, base_model: str, cfg: Dict[str, Any], run_dir: str) -> str:
        """Train and return path to the saved checkpoint (a directory)."""
        ...

    @abstractmethod
    def load_for_eval(self, model_or_path: str, cfg: Dict[str, Any]):
        """Return (model, tokenizer, device)."""
        ...
PY

cat > $PROJECT/src/methods/sft.py <<'PY'
from typing import Any, Dict
import os
import torch
from datasets import DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)

from .base import BaseMethod
from src.common.prompts import make_source, make_target_label_only

class SFTMethod(BaseMethod):
    name = "sft"

    def _build_tokenized(self, ds: DatasetDict, tok, cfg: Dict[str, Any], prompt_mode: str):
        max_in = int(cfg["generation"]["max_input_length"])
        def f(ex):
            src = make_source(ex["statement"], prompt_mode)
            tgt = make_target_label_only(ex["label_bin"])
            m = tok(src, truncation=True, max_length=max_in)
            with tok.as_target_tokenizer():
                lab = tok(tgt, truncation=True, max_length=8)
            m["labels"] = lab["input_ids"]
            return m
        return ds.map(f, remove_columns=ds["train"].column_names)

    def train(self, base_model: str, cfg: Dict[str, Any], run_dir: str) -> str:
        import math
        from src.data.liar import load_liar

        ds = load_liar(cfg)
        tok = AutoTokenizer.from_pretrained(base_model)
        model = AutoModelForSeq2SeqLM.from_pretrained(base_model)

        prompt_mode = "instruction"  # sft默认instruction
        tds = self._build_tokenized(ds, tok, cfg, prompt_mode)

        tc = DataCollatorForSeq2Seq(tok, model=model)

        tr = cfg["train"]
        args = Seq2SeqTrainingArguments(
            output_dir=os.path.join(run_dir, "ckpt"),
            learning_rate=float(tr["lr"]),
            num_train_epochs=float(tr["epochs"]),
            per_device_train_batch_size=int(tr["per_device_train_batch_size"]),
            per_device_eval_batch_size=int(tr["per_device_eval_batch_size"]),
            gradient_accumulation_steps=int(tr["grad_accum"]),
            weight_decay=float(tr["weight_decay"]),
            predict_with_generate=True,
            fp16=bool(tr["fp16"]),
            logging_steps=int(tr["logging_steps"]),
            evaluation_strategy="steps",
            eval_steps=int(tr["eval_steps"]),
            save_strategy="steps",
            save_steps=int(tr["save_steps"]),
            save_total_limit=2,
            report_to=[],
        )

        trainer = Seq2SeqTrainer(
            model=model,
            args=args,
            train_dataset=tds["train"],
            eval_dataset=tds["validation"],
            tokenizer=tok,
            data_collator=tc,
        )
        trainer.train()
        trainer.save_model(args.output_dir)
        tok.save_pretrained(args.output_dir)
        return args.output_dir

    def load_for_eval(self, model_or_path: str, cfg: Dict[str, Any]):
        tok = AutoTokenizer.from_pretrained(model_or_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_or_path)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        model.eval()
        return model, tok, device
PY

cat > $PROJECT/src/methods/soft_prompt.py <<'PY'
from typing import Any, Dict
import os
import torch
from datasets import DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)
from peft import PromptTuningConfig, get_peft_model, TaskType

from .base import BaseMethod
from src.common.prompts import make_source, make_target_label_only

class SoftPromptMethod(BaseMethod):
    name = "soft_prompt"

    def _build_tokenized(self, ds: DatasetDict, tok, cfg: Dict[str, Any], prompt_mode: str):
        max_in = int(cfg["generation"]["max_input_length"])
        def f(ex):
            src = make_source(ex["statement"], prompt_mode)
            tgt = make_target_label_only(ex["label_bin"])
            m = tok(src, truncation=True, max_length=max_in)
            with tok.as_target_tokenizer():
                lab = tok(tgt, truncation=True, max_length=8)
            m["labels"] = lab["input_ids"]
            return m
        return ds.map(f, remove_columns=ds["train"].column_names)

    def train(self, base_model: str, cfg: Dict[str, Any], run_dir: str) -> str:
        from src.data.liar import load_liar

        ds = load_liar(cfg)
        tok = AutoTokenizer.from_pretrained(base_model)
        backbone = AutoModelForSeq2SeqLM.from_pretrained(base_model)

        m = int(cfg["soft_prompt"]["m"])
        peft_cfg = PromptTuningConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            num_virtual_tokens=m,
        )
        model = get_peft_model(backbone, peft_cfg)
        model.print_trainable_parameters()

        prompt_mode = "instruction"
        tds = self._build_tokenized(ds, tok, cfg, prompt_mode)
        tc = DataCollatorForSeq2Seq(tok, model=model)

        tr = cfg["train"]
        args = Seq2SeqTrainingArguments(
            output_dir=os.path.join(run_dir, "ckpt"),
            learning_rate=float(tr["lr"]),
            num_train_epochs=float(tr["epochs"]),
            per_device_train_batch_size=int(tr["per_device_train_batch_size"]),
            per_device_eval_batch_size=int(tr["per_device_eval_batch_size"]),
            gradient_accumulation_steps=int(tr["grad_accum"]),
            weight_decay=float(tr["weight_decay"]),
            predict_with_generate=True,
            fp16=bool(tr["fp16"]),
            logging_steps=int(tr["logging_steps"]),
            evaluation_strategy="steps",
            eval_steps=int(tr["eval_steps"]),
            save_strategy="steps",
            save_steps=int(tr["save_steps"]),
            save_total_limit=2,
            report_to=[],
        )

        trainer = Seq2SeqTrainer(
            model=model,
            args=args,
            train_dataset=tds["train"],
            eval_dataset=tds["validation"],
            tokenizer=tok,
            data_collator=tc,
        )
        trainer.train()
        # peft: save adapter + tokenizer
        model.save_pretrained(args.output_dir)
        tok.save_pretrained(args.output_dir)
        return args.output_dir

    def load_for_eval(self, model_or_path: str, cfg: Dict[str, Any]):
        # PEFT adapter可以直接 AutoModelForSeq2SeqLM.from_pretrained + PeftModel.from_pretrained，但最简是让 peft 自己load
        from peft import PeftModel
        tok = AutoTokenizer.from_pretrained(model_or_path)
        # 先从adapter读取 base_model_name_or_path
        adapter_cfg_path = os.path.join(model_or_path, "adapter_config.json")
        if not os.path.exists(adapter_cfg_path):
            raise FileNotFoundError("soft_prompt checkpoint missing adapter_config.json")

        import json
        with open(adapter_cfg_path, "r", encoding="utf-8") as f:
            base = json.load(f)["base_model_name_or_path"]

        backbone = AutoModelForSeq2SeqLM.from_pretrained(base)
        model = PeftModel.from_pretrained(backbone, model_or_path)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        model.eval()
        return model, tok, device
PY

cat > $PROJECT/src/methods/cot.py <<'PY'
"""
CoT SFT: 需要 reasoning。
我们用教师模型生成 reasoning（缓存到 cache/），然后训练学生输出：
Reasoning: ...
Final label: True/False
"""
from typing import Any, Dict
import os
import torch
from datasets import DatasetDict, load_from_disk
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)

from .base import BaseMethod
from src.common.prompts import make_source, make_target_cot
from src.common.parsing import extract_binary_label
from src.common.generation import build_generate_kwargs

class CoTMethod(BaseMethod):
    name = "cot"

    def _cache_path(self, run_dir: str) -> str:
        return os.path.join(run_dir, "cot_cache_ds")

    @torch.no_grad()
    def build_cot_cache(self, base_model: str, cfg: Dict[str, Any], run_dir: str) -> str:
        """
        生成 reasoning 缓存：teacher 看到 statement + gold label，生成 1-2句 justification。
        这样不会引入 label 噪声（论文里你可写 "rationale distillation conditioned on gold label"）。
        """
        from src.data.liar import load_liar
        from src.common.prompts import make_target_label_only

        ds = load_liar(cfg)

        # choose teacher by family
        if "t5" in base_model.lower() or "flan" in base_model.lower():
            teacher = cfg["cot"]["teacher_flan"]
        else:
            teacher = cfg["cot"]["teacher_bart"]

        tok = AutoTokenizer.from_pretrained(teacher)
        model = AutoModelForSeq2SeqLM.from_pretrained(teacher)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        model.eval()

        gen_kwargs = build_generate_kwargs(cfg, for_eval=True)

        def gen_reasoning(statement: str, label_bin: int) -> str:
            gold = "True" if int(label_bin)==1 else "False"
            prompt = (
                "Task: Write a brief 1-2 sentence justification for the given claim, "
                "consistent with the provided correct label.\n"
                f'Claim: "{statement}"\n'
                f"Correct label: {gold}\n"
                "Format:\n"
                "Reasoning: <short justification>\n"
                f"Final label: {gold}"
            )
            inputs = tok(prompt, return_tensors="pt", truncation=True,
                         max_length=int(cfg["generation"]["max_input_length"])).to(device)
            out = model.generate(**inputs, **gen_kwargs)
            text = tok.decode(out[0], skip_special_tokens=True).strip()
            # 只取 Reasoning 行（若 teacher 没按格式，就把整段当 reasoning）
            if "Reasoning:" in text:
                return text.split("Final label:")[0].strip()
            return text

        def add_reasoning(split_ds):
            def f(ex):
                r = gen_reasoning(ex["statement"], ex["label_bin"])
                return {"reasoning": r}
            return split_ds.map(f)

        out = DatasetDict()
        for sp in ["train", "validation", "test"]:
            out[sp] = add_reasoning(ds[sp])

        cache_dir = self._cache_path(run_dir)
        out.save_to_disk(cache_dir)
        return cache_dir

    def _build_tokenized(self, ds: DatasetDict, tok, cfg: Dict[str, Any], prompt_mode: str):
        max_in = int(cfg["generation"]["max_input_length"])
        def f(ex):
            src = make_source(ex["statement"], prompt_mode)
            tgt = make_target_cot(ex.get("reasoning",""), ex["label_bin"])
            m = tok(src, truncation=True, max_length=max_in)
            with tok.as_target_tokenizer():
                lab = tok(tgt, truncation=True, max_length=int(cfg["generation"]["max_new_tokens"]))
            m["labels"] = lab["input_ids"]
            return m
        return ds.map(f, remove_columns=ds["train"].column_names)

    def train(self, base_model: str, cfg: Dict[str, Any], run_dir: str) -> str:
        tok = AutoTokenizer.from_pretrained(base_model)
        model = AutoModelForSeq2SeqLM.from_pretrained(base_model)

        # build cache
        cache_dir = self._cache_path(run_dir)
        if not os.path.exists(cache_dir):
            self.build_cot_cache(base_model, cfg, run_dir)
        ds = load_from_disk(cache_dir)

        prompt_mode = "cot"
        tds = self._build_tokenized(ds, tok, cfg, prompt_mode)
        tc = DataCollatorForSeq2Seq(tok, model=model)

        tr = cfg["train"]
        args = Seq2SeqTrainingArguments(
            output_dir=os.path.join(run_dir, "ckpt"),
            learning_rate=float(tr["lr"]),
            num_train_epochs=float(tr["epochs"]),
            per_device_train_batch_size=int(tr["per_device_train_batch_size"]),
            per_device_eval_batch_size=int(tr["per_device_eval_batch_size"]),
            gradient_accumulation_steps=int(tr["grad_accum"]),
            weight_decay=float(tr["weight_decay"]),
            predict_with_generate=True,
            fp16=bool(tr["fp16"]),
            logging_steps=int(tr["logging_steps"]),
            evaluation_strategy="steps",
            eval_steps=int(tr["eval_steps"]),
            save_strategy="steps",
            save_steps=int(tr["save_steps"]),
            save_total_limit=2,
            report_to=[],
        )

        trainer = Seq2SeqTrainer(
            model=model,
            args=args,
            train_dataset=tds["train"],
            eval_dataset=tds["validation"],
            tokenizer=tok,
            data_collator=tc,
        )
        trainer.train()
        trainer.save_model(args.output_dir)
        tok.save_pretrained(args.output_dir)
        return args.output_dir

    def load_for_eval(self, model_or_path: str, cfg: Dict[str, Any]):
        tok = AutoTokenizer.from_pretrained(model_or_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_or_path)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        model.eval()
        return model, tok, device
PY

cat > $PROJECT/src/methods/registry.py <<'PY'
from .sft import SFTMethod
from .cot import CoTMethod
from .soft_prompt import SoftPromptMethod

def get_method(name: str):
    n = str(name).strip().lower()
    if n in ["sft", "instruction_sft"]:
        return SFTMethod()
    if n in ["cot", "cot_sft", "instruction_cot"]:
        return CoTMethod()
    if n in ["soft_prompt", "prompt", "prompt_tuning"]:
        return SoftPromptMethod()
    raise ValueError(f"Unknown method: {name}")
PY

cat > $PROJECT/src/eval/evaluator.py <<'PY'
from typing import Any, Dict, List
import os
import torch

from src.common.prompts import make_source
from src.common.parsing import extract_binary_label
from src.common.generation import build_generate_kwargs
from src.common.metrics import macro_f1, accuracy, measure_latency_ms

class Evaluator:
    def __init__(self, cfg: Dict[str, Any]):
        self.cfg = cfg

    @torch.no_grad()
    def evaluate(self, model, tok, device: str, dataset, split: str, prompt_mode: str) -> Dict[str, Any]:
        limit = int(self.cfg["data"].get("limit_eval", 0))
        rows = dataset[split]
        if limit > 0:
            rows = rows.select(range(min(limit, len(rows))))

        gen_kwargs = build_generate_kwargs(self.cfg, for_eval=True)

        true_labels: List[int] = []
        pred_labels: List[int] = []
        unparsable = 0
        prompts_for_latency: List[str] = []

        def generate_one(prompt: str) -> str:
            inputs = tok(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=int(self.cfg["generation"]["max_input_length"]),
            ).to(device)
            out = model.generate(**inputs, **gen_kwargs)
            return tok.decode(out[0], skip_special_tokens=True)

        for ex in rows:
            prompt = make_source(ex["statement"], prompt_mode)
            prompts_for_latency.append(prompt)

            text = generate_one(prompt)
            pred, how = extract_binary_label(text)
            true = int(ex["label_bin"])

            true_labels.append(true)
            if pred is None:
                unparsable += 1
                pred_labels.append(0)  # 占位：主表会看 unparsable_rate
            else:
                pred_labels.append(int(pred))

        return dict(
            split=split,
            n=len(true_labels),
            prompt_mode=prompt_mode,
            macro_f1=macro_f1(true_labels, pred_labels),
            accuracy=accuracy(true_labels, pred_labels),
            mean_latency_ms=measure_latency_ms(
                generate_one, prompts_for_latency, n_samples=min(200, len(prompts_for_latency))
            ),
            unparsable_rate=unparsable / max(1, len(true_labels)),
            generation=gen_kwargs,
        )
PY

cat > $PROJECT/run.py <<'PY'
import argparse, os
from datetime import datetime

from src.common.io import read_yaml, ensure_dir, write_json, dir_size_mb
from src.common.seed import set_seed
from src.data.liar import load_liar
from src.methods.registry import get_method
from src.eval.evaluator import Evaluator

def default_prompt_for_method(method: str) -> str:
    m = method.lower()
    if m in ["cot", "cot_sft", "instruction_cot"]:
        return "cot"
    return "instruction"

def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)

    ap_train = sub.add_parser("train")
    ap_train.add_argument("--config", default="configs/base.yaml")
    ap_train.add_argument("--method", required=True, choices=["sft","cot","soft_prompt"])
    ap_train.add_argument("--base_model", required=True)
    ap_train.add_argument("--run_name", default="")

    ap_eval = sub.add_parser("eval")
    ap_eval.add_argument("--config", default="configs/base.yaml")
    ap_eval.add_argument("--method", required=True, choices=["sft","cot","soft_prompt"])
    ap_eval.add_argument("--model_or_path", required=True)
    ap_eval.add_argument("--split", default="validation", choices=["train","validation","test"])
    ap_eval.add_argument("--prompt_mode", default="")
    ap_eval.add_argument("--run_name", default="")

    args = ap.parse_args()

    root = os.path.dirname(__file__)
    cfg_path = args.config if os.path.isabs(args.config) else os.path.join(root, args.config)
    cfg = read_yaml(cfg_path)
    set_seed(int(cfg.get("seed", 42)))

    method = get_method(args.method)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    if args.cmd == "train":
        run_name = args.run_name.strip() or f"train_{args.method}_{args.base_model.replace('/','-')}_{ts}"
        run_dir = ensure_dir(os.path.join(root, "artifacts", run_name))

        ckpt_dir = method.train(args.base_model, cfg, run_dir)

        write_json(os.path.join(run_dir, "manifest.json"), dict(
            cmd="train",
            run_name=run_name,
            method=args.method,
            base_model=args.base_model,
            ckpt_dir=ckpt_dir,
            config_path=args.config,
            artifacts_size_mb=dir_size_mb(run_dir),
        ))
        print("TRAIN DONE:", run_dir)
        print("CKPT:", ckpt_dir)
        return

    if args.cmd == "eval":
        run_name = args.run_name.strip() or f"eval_{args.method}_{args.model_or_path.replace('/','-').replace('.','_')}_{args.split}_{ts}"
        run_dir = ensure_dir(os.path.join(root, "artifacts", run_name))

        ds = load_liar(cfg)
        model, tok, device = method.load_for_eval(args.model_or_path, cfg)

        prompt_mode = args.prompt_mode.strip().lower() or default_prompt_for_method(args.method)

        evaluator = Evaluator(cfg)
        result = evaluator.evaluate(model, tok, device, ds, split=args.split, prompt_mode=prompt_mode)

        write_json(os.path.join(run_dir, "result.json"), result)
        write_json(os.path.join(run_dir, "manifest.json"), dict(
            cmd="eval",
            run_name=run_name,
            method=args.method,
            model_or_path=args.model_or_path,
            split=args.split,
            prompt_mode=prompt_mode,
            config_path=args.config,
            artifacts_size_mb=dir_size_mb(run_dir),
        ))
        print("EVAL DONE:", run_dir)
        print(result)
        return

if __name__ == "__main__":
    main()
PY

echo "Created: $PROJECT"
ls -R $PROJECT | sed -n '1,180p'


Created: /content/liar_project_v2
/content/liar_project_v2:
artifacts
cache
configs
run.py
src

/content/liar_project_v2/artifacts:

/content/liar_project_v2/cache:

/content/liar_project_v2/configs:
base.yaml

/content/liar_project_v2/src:
common
data
eval
methods

/content/liar_project_v2/src/common:
generation.py
io.py
metrics.py
parsing.py
prompts.py
seed.py

/content/liar_project_v2/src/data:
liar.py

/content/liar_project_v2/src/eval:
evaluator.py

/content/liar_project_v2/src/methods:
base.py
cot.py
registry.py
sft.py
soft_prompt.py


In [3]:
%cd /content/liar_project_v2
!python run.py eval --method sft --model_or_path google/flan-t5-small --split validation
!python run.py eval --method sft --model_or_path facebook/bart-base --split validation

/content/liar_project_v2
2025-11-28 16:04:01.313366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764345841.347028    1932 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764345841.356702    1932 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764345841.387791    1932 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764345841.387840    1932 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764345841.387849    1932 computation_placer.cc:17

In [4]:
!python run.py train --method sft --base_model google/flan-t5-small
# 训练完会输出 CKPT 目录，把它复制下来，然后：
!python run.py eval --method sft --model_or_path /content/liar_project_v2/artifacts/<这里换成你刚生成的run_name>/ckpt --split validation


2025-11-28 16:07:16.704723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764346036.725209    2896 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764346036.731249    2896 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764346036.746714    2896 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764346036.746757    2896 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764346036.746761    2896 computation_placer.cc:177] computation placer alr